# QuantIQ | Phase 2 — Market Analysis

**Phase:** 2 | **Weeks:** 4–6 | **Deadline:** 14 June 2026

**Purpose:** Individual stock analysis for 12 NIFTY50 stocks — technical indicators, fundamentals, and cross-sector correlation.

---

## Team

| Handle | Stock          | Sector                |
|--------|----------------|-----------------------|
| AJ     | RELIANCE.NS    | Energy / Conglomerate |
| AV     | TCS.NS         | IT                    |
| AK     | INFY.NS        | IT                    |
| SS     | HCLTECH.NS     | IT                    |
| AR     | HDFCBANK.NS    | Banking               |
| EB     | ICICIBANK.NS   | Banking               |
| ShS    | AXISBANK.NS    | Banking               |
| RS     | TVSMOTOR.NS         | Auto                  |
| GT     | M&M.NS         | Auto                  |
| RT     | LT.NS          | Infrastructure        |
| NS     | TITAN.NS       | Consumer              |
| SmS    | APOLLOMICRO.NS | Defence               |

**Section 13 — Correlation Heatmap:** RS + GT

---

*Created by RS (Project Lead). Do not modify header cells or shared helper cells.*

In [2]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from src.data import fetch_ohlc

pio.templates["plotly_dark"].layout.paper_bgcolor = "#252526"
pio.templates["plotly_dark"].layout.plot_bgcolor  = "#1F2531"
pio.templates.default = "plotly_dark"

print("Imports OK")

Imports OK


In [3]:
TICKERS = {
    "AJ":  "RELIANCE.NS",
    "AV":  "TCS.NS",
    "AK":  "INFY.NS",
    "SS":  "HCLTECH.NS",
    "AR":  "HDFCBANK.NS",
    "EB":  "ICICIBANK.NS",
    "ShS": "AXISBANK.NS",
    "RS":  "TVSMOTOR.NS",
    "GT":  "M&M.NS",
    "RT":  "LT.NS",
    "NS":  "TITAN.NS",
    "SmS": "APOLLOMICRO.NS",
}

PERIOD   = "1y"
INTERVAL = "1d"

print(f"Config loaded — {len(TICKERS)} tickers, period={PERIOD}, interval={INTERVAL}")

Config loaded — 12 tickers, period=1y, interval=1d


In [4]:
def fetch_stock(
    ticker: str,
    period: str = PERIOD,
    interval: str = INTERVAL,
) -> pd.DataFrame:
    """Fetch OHLCV for a .NS ticker via fetch_ohlc.

    Args:
        ticker (str): NSE ticker with .NS suffix (e.g. "RELIANCE.NS").
        period (str): yfinance period string. Default "1y".
        interval (str): yfinance interval string. Default "1d".

    Returns:
        pd.DataFrame: OHLCV DataFrame indexed by datetime, NaN rows dropped.

    Raises:
        AssertionError: If ticker does not end with .NS.
        ValueError: If no data returned for ticker.
    """
    assert ticker.endswith(".NS"), f"Ticker must have .NS suffix, got: {ticker}"
    df = fetch_ohlc(ticker, period=period, interval=interval, use_cache=True)
    if df is None or df.empty:
        raise ValueError(f"No data returned for {ticker}")
    return df.dropna()

## Energy

## Section 1 — RELIANCE.NS | AJ (Energy / Conglomerate)

In [5]:
# ── 1a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

RELIANCE.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,1301.681536,1321.689127,1294.315690,1308.549805,23302785
2026-06-03,1308.948015,1317.906594,1295.012446,1307.156250,20012293
2026-06-04,1295.012426,1305.165434,1287.148760,1297.699951,23474327
2026-06-05,1304.500000,1306.000000,1288.000000,1291.000000,17785223
2026-06-08,1277.000000,1282.599976,1259.199951,1263.300049,16490817


In [6]:
# ── 1b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,1424.06,1435.67,1412.02,1423.05,13012915.60
std,70.09,69.45,70.70,70.11,6886857.07
min,1277.00,1282.60,1259.20,1263.30,0.00
25%,1371.46,1382.11,1361.70,1370.76,8195700.00
50%,1413.46,1421.43,1397.14,1407.09,11135399.00
75%,1477.07,1485.93,1463.33,1472.88,16683858.00
max,1585.67,1604.38,1570.94,1584.97,42634247.00


In [7]:
# ── 1c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [8]:
# ── 1d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [9]:
# ── 1e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [10]:
# ── 1f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [11]:
# ── 1g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AJ"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 21.20
  EPS Growth (TTM) : -12.60%
  Debt / Equity    : 36.65
  Total Cash       : ₹2,581,470,117,888
  Total Debt       : ₹3,979,999,969,280
  Current Ratio    : 1.10
  PEG Ratio        : 0.82
  Dividend Yield   : 47.00%


### Observations

_TODO (AJ): Write 3–5 paragraphs on what the technical and fundamental data reveals about RELIANCE.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## IT

## Section 2 — TCS.NS | AV (IT)

In [12]:
# ── 2a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TCS.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,2320.000000,2457.399902,2316.199951,2446.899902,11306127
2026-06-03,2393.000000,2393.000000,2224.800049,2241.699951,15660062
2026-06-04,2241.699951,2253.100098,2216.600098,2241.000000,4867263
2026-06-05,2262.899902,2271.800049,2192.000000,2198.899902,5019717
2026-06-08,2170.000000,2177.000000,2143.300049,2151.399902,6534600


In [13]:
# ── 2b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,2865.29,2887.26,2836.93,2859.34,3417583.51
std,321.74,318.91,322.69,322.64,2122478.82
min,2170.00,2177.00,2143.30,2151.40,0.00
25%,2550.42,2579.03,2525.75,2546.97,2271789.00
50%,2954.14,2972.87,2932.54,2952.56,2899519.00
75%,3105.67,3125.05,3082.60,3100.69,3924009.00
max,3385.87,3404.06,3356.04,3382.21,16331582.00


In [14]:
# ── 2c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [15]:
# ── 2d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [16]:
# ── 2e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [17]:
# ── 2f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [18]:
# ── 2g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AV"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.82
  EPS Growth (TTM) : 12.20%
  Debt / Equity    : 10.39
  Total Cash       : ₹410,799,996,928
  Total Debt       : ₹112,829,997,056
  Current Ratio    : 2.23
  PEG Ratio        : 2.38
  Dividend Yield   : 576.00%


### Observations

_TODO (AV): Write 3–5 paragraphs on what the technical and fundamental data reveals about TCS.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 3 — INFY.NS | AK (IT)

In [19]:
# ── 3a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

INFY.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,1232.500000,1278.900024,1232.500000,1270.800049,36032600
2026-06-03,1242.099976,1249.900024,1216.000000,1222.599976,18534303
2026-06-04,1208.000000,1215.199951,1196.199951,1201.300049,12755957
2026-06-05,1220.000000,1223.800049,1194.300049,1197.500000,8692290
2026-06-08,1177.500000,1200.699951,1176.500000,1187.599976,8630281


In [20]:
# ── 3b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,1448.34,1461.93,1433.34,1446.83,10113122.36
std,154.51,152.93,153.80,154.23,7251968.60
min,1102.00,1121.90,1089.00,1095.00,0.00
25%,1313.00,1325.00,1299.00,1312.60,6224582.00
50%,1483.98,1491.57,1469.00,1481.62,8294615.00
75%,1568.09,1586.00,1556.28,1577.84,12096377.00
max,1727.20,1728.00,1666.50,1689.80,69085430.00


In [21]:
# ── 3c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [22]:
# ── 3d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [23]:
# ── 3e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [24]:
# ── 3f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [25]:
# ── 3g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AK"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.47
  EPS Growth (TTM) : 11.80%
  Debt / Equity    : 9.83
  Total Cash       : ₹3,705,999,872
  Total Debt       : ₹967,000,000
  Current Ratio    : 1.98
  PEG Ratio        : 2.21
  Dividend Yield   : 421.00%


### Observations

_TODO (AK): Write 3–5 paragraphs on what the technical and fundamental data reveals about INFY.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 4 — HCLTECH.NS | SS (IT)

In [26]:
# ── 4a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HCLTECH.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,1210.000000,1257.0,1202.199951,1243.500000,7663067
2026-06-03,1228.800049,1230.0,1176.800049,1179.000000,4800922
2026-06-04,1166.400024,1176.5,1158.199951,1168.300049,2025595
2026-06-05,1179.000000,1182.5,1147.000000,1154.699951,1971395
2026-06-08,1141.900024,1159.0,1132.300049,1151.300049,1545087


In [27]:
# ── 4b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,1456.11,1470.11,1440.30,1454.52,3193621.95
std,150.50,149.90,149.34,150.79,2730849.59
min,1121.90,1143.20,1103.40,1124.00,0.00
25%,1364.87,1380.66,1344.13,1355.92,1936466.00
50%,1447.63,1458.75,1435.17,1444.54,2568054.00
75%,1589.83,1610.91,1581.28,1595.37,3459049.00
max,1746.56,1746.66,1668.56,1697.11,33066256.00


In [28]:
# ── 4c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [29]:
# ── 4d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [30]:
# ── 4e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [31]:
# ── 4f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [32]:
# ── 4g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 18.62
  EPS Growth (TTM) : -0.20%
  Debt / Equity    : 6.93
  Total Cash       : ₹3,204,999,936
  Total Debt       : ₹550,000,000
  Current Ratio    : 2.22
  PEG Ratio        : 2.39
  Dividend Yield   : 834.00%


### Observations

_TODO (SS): Write 3–5 paragraphs on what the technical and fundamental data reveals about HCLTECH.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Banking

## Section 5 — HDFCBANK.NS | AR (Banking)

In [33]:
# ── 5a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HDFCBANK.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,737.000000,753.599976,733.150024,748.250000,47314447
2026-06-03,744.450012,756.900024,742.599976,753.650024,36109480
2026-06-04,749.150024,757.299988,745.000000,754.200012,42673538
2026-06-05,753.950012,758.700012,744.650024,747.049988,22116568
2026-06-08,738.000000,741.500000,734.500000,738.650024,21546501


In [34]:
# ── 5b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,2.490000e+02
mean,923.60,931.26,917.05,923.87,2.703903e+07
std,88.71,87.27,89.00,88.04,1.865332e+07
min,730.00,741.50,726.65,731.55,0.000000e+00
25%,850.05,871.70,848.00,857.05,1.515404e+07
50%,962.70,969.35,956.00,963.40,2.211657e+07
75%,991.25,998.52,986.00,992.10,3.462076e+07
max,1017.50,1020.50,1008.50,1012.90,1.716374e+08


In [35]:
# ── 5c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [36]:
# ── 5d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [37]:
# ── 5e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [38]:
# ── 5f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [39]:
# ── 5g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["AR"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.38
  EPS Growth (TTM) : 7.50%
  Debt / Equity    : N/A
  Total Cash       : ₹3,119,260,631,040
  Total Debt       : ₹5,884,845,490,176
  Current Ratio    : N/A
  PEG Ratio        : 0.89
  Dividend Yield   : 176.00%


### Observations

_TODO (AR): Write 3–5 paragraphs on what the technical and fundamental data reveals about HDFCBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 6 — ICICIBANK.NS | EB (Banking)

In [40]:
# ── 6a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

ICICIBANK.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,1233.000000,1237.699951,1220.699951,1226.599976,17920892
2026-06-03,1219.900024,1248.300049,1213.699951,1242.000000,13934754
2026-06-04,1232.199951,1262.300049,1232.199951,1251.699951,19461560
2026-06-05,1257.000000,1265.000000,1250.000000,1262.099976,10166393
2026-06-08,1246.400024,1254.000000,1243.099976,1250.199951,10713430


In [41]:
# ── 6b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,1365.32,1375.82,1355.79,1365.81,12844360.59
std,64.16,62.38,64.59,63.54,6377717.67
min,1196.00,1222.10,1187.60,1205.90,0.00
25%,1340.50,1352.90,1334.00,1343.40,7805252.00
50%,1377.00,1387.60,1370.30,1379.00,11499836.00
75%,1411.11,1418.85,1401.90,1412.30,17037215.00
max,1488.51,1488.51,1466.78,1477.20,38929053.00


In [42]:
# ── 6c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [43]:
# ── 6d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [44]:
# ── 6e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [45]:
# ── 6f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [46]:
# ── 6g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["EB"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 16.95
  EPS Growth (TTM) : 8.40%
  Debt / Equity    : N/A
  Total Cash       : ₹2,649,807,126,528
  Total Debt       : ₹2,202,642,677,760
  Current Ratio    : N/A
  PEG Ratio        : 0.53
  Dividend Yield   : 88.00%


### Observations

_TODO (EB): Write 3–5 paragraphs on what the technical and fundamental data reveals about ICICIBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 7 — AXISBANK.NS | ShS (Banking)

In [47]:
# ── 7a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

AXISBANK.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,1263.000000,1275.800049,1244.900024,1251.099976,11884695
2026-06-03,1245.099976,1264.800049,1230.300049,1255.199951,7953326
2026-06-04,1246.000000,1259.900024,1244.099976,1253.300049,6392593
2026-06-05,1260.000000,1276.000000,1252.099976,1272.300049,7825303
2026-06-08,1262.000000,1279.199951,1262.000000,1268.099976,5033696


In [48]:
# ── 7b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,1227.42,1239.11,1216.43,1227.65,6679355.94
std,90.40,91.68,88.21,90.07,3962885.03
min,1047.60,1058.00,1042.50,1045.20,0.00
25%,1174.60,1180.00,1163.00,1170.80,4311076.00
50%,1235.84,1246.00,1225.00,1237.30,5822618.00
75%,1288.80,1298.80,1273.00,1286.60,8088417.00
max,1403.00,1418.30,1387.60,1403.00,38053695.00


In [49]:
# ── 7c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [50]:
# ── 7d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [51]:
# ── 7e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [52]:
# ── 7f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [53]:
# ── 7g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["ShS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 15.17
  EPS Growth (TTM) : 6.40%
  Debt / Equity    : N/A
  Total Cash       : ₹1,093,515,673,600
  Total Debt       : ₹2,805,106,212,864
  Current Ratio    : N/A
  PEG Ratio        : 0.54
  Dividend Yield   : 8.00%


### Observations

_TODO (ShS): Write 3–5 paragraphs on what the technical and fundamental data reveals about AXISBANK.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Auto

## Section 8 — TVSMOTOR.NS | RS (Auto)

In [54]:
# ── 8a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TVSMOTOR.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,3328.0,3383.000000,3291.600098,3366.899902,516666
2026-06-03,3360.0,3366.800049,3301.399902,3316.899902,846948
2026-06-04,3310.0,3400.699951,3292.199951,3362.500000,1089659
2026-06-05,3380.0,3397.000000,3345.000000,3383.899902,813580
2026-06-08,3325.0,3378.899902,3301.000000,3317.600098,949238


In [55]:
# ── 8b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,3420.34,3456.16,3378.36,3418.30,847999.45
std,322.46,324.68,316.23,320.55,540024.18
min,2695.58,2738.23,2645.85,2729.86,0.00
25%,3325.00,3378.90,3275.00,3317.60,516666.00
50%,3491.80,3519.70,3453.93,3487.71,727156.00
75%,3647.05,3674.36,3592.60,3641.27,1025395.00
max,3940.43,3956.17,3904.45,3940.43,3748828.00


In [56]:
# ── 8c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [57]:
# ── 8d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [58]:
# ── 8e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [59]:
# ── 8f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [60]:
# ── 8g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 52.98
  EPS Growth (TTM) : 19.10%
  Debt / Equity    : 306.09
  Total Cash       : ₹50,211,201,024
  Total Debt       : ₹327,910,096,896
  Current Ratio    : 0.98
  PEG Ratio        : N/A
  Dividend Yield   : 36.00%


### Observations

_TODO (RS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TVSMOTOR.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 9 — M&M.NS | GT (Auto)

In [61]:
# ── 9a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

M&M.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,2913.100098,3008.699951,2913.100098,2998.300049,4045354
2026-06-03,2996.000000,3029.500000,2965.000000,3011.100098,3110137
2026-06-04,2985.000000,3067.100098,2974.199951,3016.100098,2788572
2026-06-05,3039.899902,3059.000000,3012.699951,3040.500000,1685633
2026-06-08,2985.000000,3006.000000,2948.000000,2966.000000,2415345


In [62]:
# ── 9b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,3374.41,3410.91,3337.97,3372.62,2572835.85
std,246.37,242.01,245.89,243.07,1428399.47
min,2902.00,2994.44,2896.00,2931.10,0.00
25%,3150.90,3192.35,3128.56,3159.02,1701384.00
50%,3379.00,3420.00,3335.30,3384.40,2260853.00
75%,3608.20,3640.20,3577.90,3604.40,3041614.00
max,3805.90,3839.90,3785.00,3802.40,11402867.00


In [63]:
# ── 9c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [64]:
# ── 9d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [65]:
# ── 9e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [66]:
# ── 9f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [67]:
# ── 9g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["GT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 19.78
  EPS Growth (TTM) : 44.80%
  Debt / Equity    : 125.07
  Total Cash       : ₹527,834,513,408
  Total Debt       : ₹1,369,359,122,432
  Current Ratio    : 1.27
  PEG Ratio        : 1.26
  Dividend Yield   : 111.00%


### Observations
Mahindra & Mahindra exhibits a resilient core architecture driven by its dominant positioning in the utility vehicles segment and aggressive capital allocation toward EV industrialization via its 'Born Electric' platform, creating a highly scalable valuation expansion layer. From a fundamental standpoint, M&M displays strong balance sheet mechanics characterized by robust return ratios, positive free cash flow generation, a low debt-to-equity posture, and an attractive forward P/E multiple relative to its growth ($PEG < 1.0$). Technically, the stock is trading within a well-defined secular ascending channel on the weekly timeframes above its key moving averages ($EMA_{50}$ and $EMA_{200}$), supported by a firm Relative Strength Index (RSI) in the constructive 60–65 zone and a bullish MACD histogram expansion. Backed by stellar monthly wholesale dispatch numbers and strategic partnerships for global battery supply chains, M&M reflects high institutional inflows and strong market optimism, consolidating it as a high-conviction thematic bet across both defensive and cyclical automotive sectors.

## Infrastructure

## Section 10 — LT.NS | RT (Infrastructure)

In [68]:
# ── 10a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

LT.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,3980.000000,4017.000000,3923.500000,4000.899902,2085054
2026-06-03,4000.899902,4008.000000,3894.000000,3953.199951,1872929
2026-06-04,3945.000000,3974.899902,3930.100098,3942.100098,1398297
2026-06-05,3960.000000,3978.000000,3923.899902,3953.199951,1278243
2026-06-08,3901.100098,3916.699951,3863.100098,3875.500000,1678526


In [69]:
# ── 10b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,3808.53,3840.19,3772.38,3805.56,2167692.51
std,225.72,227.66,225.71,227.78,1548960.19
min,3372.16,3376.92,3256.29,3310.07,0.00
25%,3600.83,3633.51,3562.50,3593.60,1251770.00
50%,3842.17,3896.74,3809.79,3844.95,1749063.00
75%,3993.00,4026.47,3961.50,3992.11,2335718.00
max,4372.79,4397.05,4329.41,4375.36,10778730.00


In [70]:
# ── 10c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [71]:
# ── 10d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [72]:
# ── 10e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [73]:
# ── 10f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [74]:
# ── 10g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["RT"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 33.41
  EPS Growth (TTM) : -34.50%
  Debt / Equity    : 97.64
  Total Cash       : ₹766,146,510,848
  Total Debt       : ₹1,254,965,903,360
  Current Ratio    : 1.25
  PEG Ratio        : 1.06
  Dividend Yield   : 98.00%


### Observations

_TODO (RT): Write 3–5 paragraphs on what the technical and fundamental data reveals about LT.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Consumer

## Section 11 — TITAN.NS | NS (Consumer)

In [75]:
# ── 11a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

TITAN.NS | 249 rows | 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-02,4017.399902,4088.000000,3986.000000,4078.100098,522015
2026-06-03,4057.000000,4103.799805,4002.000000,4088.800049,959352
2026-06-04,4075.000000,4273.799805,4052.300049,4231.000000,1761933
2026-06-05,4269.000000,4289.000000,4223.200195,4260.200195,1390200
2026-06-08,4196.799805,4238.100098,4180.700195,4192.399902,666690


In [76]:
# ── 11b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

Shape     : (249, 5)
Date range: 2025-06-09 → 2026-06-08


Price,Open,High,Low,Close,Volume
count,249.00,249.00,249.00,249.00,249.00
mean,3872.71,3910.56,3835.44,3876.26,996125.67
std,331.72,338.78,323.25,332.76,850944.81
min,3315.00,3347.30,3303.10,3316.00,0.00
25%,3555.00,3580.00,3526.40,3565.60,567278.00
50%,3875.00,3914.50,3843.50,3885.80,770589.00
75%,4143.00,4179.00,4091.00,4144.00,1113050.00
max,4554.00,4605.00,4467.70,4525.90,7524068.00


In [77]:
# ── 11c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [78]:
# ── 11d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [79]:
# ── 11e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [80]:
# ── 11f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [81]:
# ── 11g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["NS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

  P/E Ratio        : 72.37
  EPS Growth (TTM) : 35.10%
  Debt / Equity    : 195.00
  Total Cash       : ₹41,659,998,208
  Total Debt       : ₹306,210,013,184
  Current Ratio    : 1.28
  PEG Ratio        : N/A
  Dividend Yield   : 26.00%


### Observations

_TODO (NS): Write 3–5 paragraphs on what the technical and fundamental data reveals about TITAN.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Defence

## Section 12 — APOLLOMICRO.NS | SmS (Defence)

In [82]:
# ── 12a. Fetch ──────────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
df     = fetch_stock(ticker)
print(f"{ticker} | {df.shape[0]} rows | {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: APOLLOMICRO.NS"}}}
$APOLLOMICRO.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['APOLLOMICRO.NS']: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned empty DataFrame for APOLLOMICRO.NS (period=1y)


ValueError: No data returned for APOLLOMICRO.NS

In [ ]:
# ── 12b. Summary Statistics ──────────────────────────────────────────────────
print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.describe().round(2)

In [ ]:
# ── 12c. Price: Candlestick + EMA20 / EMA50 ────────────────────────────────
ema20 = EMAIndicator(close=df["Close"], window=20).ema_indicator()
ema50 = EMAIndicator(close=df["Close"], window=50).ema_indicator()

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Candlestick(
    x=df.index, open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="OHLC",
))
fig.add_trace(go.Scatter(x=df.index, y=ema20, name="EMA20", line=dict(color="orange", width=1.5)))
fig.add_trace(go.Scatter(x=df.index, y=ema50, name="EMA50", line=dict(color="#4A90D9", width=1.5)))
fig.update_layout(
    title=f"{ticker} — Price + EMA20/50",
    xaxis_rangeslider_visible=False,
    height=500,
)
fig.show()

In [ ]:
# ── 12d. MACD (12/26/9) ────────────────────────────────────────────────────
macd_ind    = MACD(close=df["Close"], window_slow=26, window_fast=12, window_sign=9)
macd_line   = macd_ind.macd()
signal_line = macd_ind.macd_signal()
histogram   = macd_ind.macd_diff()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[f"{ticker} — Close", "MACD (12/26/9)"],
)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close",  line=dict(color="white")),  row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=macd_line,   name="MACD",   line=dict(color="cyan")),   row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=signal_line, name="Signal", line=dict(color="orange")), row=2, col=1)
fig.add_trace(go.Bar(    x=df.index, y=histogram,   name="Hist",   marker_color="white"),       row=2, col=1)
fig.update_layout(height=600, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── 12e. RSI (14) ───────────────────────────────────────────────────────────
rsi = RSIIndicator(close=df["Close"], window=14).rsi()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=rsi, name="RSI(14)", line=dict(color="mediumpurple", width=1.5)))
fig.add_hline(y=70, line_dash="dash", line_color="red",       annotation_text="Overbought (70)")
fig.add_hline(y=30, line_dash="dash", line_color="limegreen", annotation_text="Oversold (30)")
fig.update_layout(
    title=f"{ticker} — RSI (14)",
    yaxis=dict(range=[0, 100]),
    height=350,
)
fig.show()

In [ ]:
# ── 12f. Volume ──────────────────────────────────────────────────────────────
vol_avg = df["Volume"].rolling(20).mean()

fig = go.Figure()
fig.add_trace(go.Bar(   x=df.index, y=df["Volume"], name="Volume",  marker_color="white", opacity=0.6))
fig.add_trace(go.Scatter(x=df.index, y=vol_avg,     name="20d Avg", line=dict(color="orange", width=1.5)))
fig.update_layout(title=f"{ticker} — Volume + 20-day Avg", height=350)
fig.show()

In [ ]:
# ── 12g. Fundamentals ─────────────────────────────────────────────────────────
ticker = TICKERS["SmS"]
info   = yf.Ticker(ticker).info

pe    = info.get("trailingPE");     pe_str  = f"{pe:.2f}"    if pe    is not None else "N/A"
eps_g = info.get("earningsGrowth"); eps_str = f"{eps_g:.2%}" if eps_g is not None else "N/A"
de    = info.get("debtToEquity");   de_str  = f"{de:.2f}"    if de    is not None else "N/A"
cash  = info.get("totalCash");      c_str   = f"{cash:,.0f}" if cash  is not None else "N/A"
debt  = info.get("totalDebt");      d_str   = f"{debt:,.0f}" if debt  is not None else "N/A"
icr   = info.get("currentRatio");   icr_str = f"{icr:.2f}"   if icr   is not None else "N/A"
peg   = info.get("pegRatio");       peg_str = f"{peg:.2f}"   if peg   is not None else "N/A"
div_y = info.get("dividendYield");  div_str = f"{div_y:.2%}" if div_y is not None else "N/A"

print(f"  P/E Ratio        : {pe_str}")
print(f"  EPS Growth (TTM) : {eps_str}")
print(f"  Debt / Equity    : {de_str}")
print(f"  Total Cash       : ₹{c_str}")
print(f"  Total Debt       : ₹{d_str}")
print(f"  Current Ratio    : {icr_str}")
print(f"  PEG Ratio        : {peg_str}")
print(f"  Dividend Yield   : {div_str}")

### Observations

_TODO (SmS): Write 3–5 paragraphs on what the technical and fundamental data reveals about APOLLOMICRO.NS. Consider: trend direction, momentum signals (RSI / MACD), valuation (P/E, PEG), and financial health (D/E, cash position)._

## Section 13 — Cross-Sector Correlation Heatmap | RS + GT

In [ ]:
# ==============================================================================
# ─── SECTION 13: CROSS-SECTOR CORRELATION HEATMAP (GT + RS) ───────────────────
# ==============================================================================
# NOTE: This block is pre-structured to run dynamically. It implements a defensive
# processing pipeline that filters out asset anomalies or schema mismatches automatically.

import numpy as np
import pandas as pd
import plotly.express as px


def generate_correlation_matrix(tickers_dict: dict, period: str = "1y", interval: str = "1d") -> None:
    """
    Defensively fetches close prices for all mapped NIFTY50 tickers, calculates
    aligned daily log returns, and renders an interactive cross-sector correlation heatmap.
    """
    close_series_map = {}

    print("Executing defensive correlation data gathering matrix...")

    for handle, ticker in tickers_dict.items():
        try:
            # Defensive Fetch: Utilizing the established safe fetch tracking wrapper
            df = fetch_stock(ticker, period=period, interval=interval)

            if df is not None and not df.empty and "Close" in df.columns:
                # Synchronize data arrays by stripping potential duplicate indices
                close_series_map[f"{handle} ({ticker.replace('.NS', '')})"] = df["Close"]
            else:
                print(f"[WARNING] Ticker {ticker} returned empty or missing 'Close' vector. Skipping matrix inclusion.")
        except Exception as e:
            print(f"[ERROR] Critical failure processing data array for handle {handle} ({ticker}): {str(e)}")
            continue

    # 1. Verification Constraint Guard
    if len(close_series_map) < 2:
        raise ValueError("Insufficient data fields captured. A minimum of 2 valid asset series are required to map a matrix.")

    # 2. Construction & Time-Series Alignment
    # Combined via an inner join to completely mitigate missing/unaligned trading day indices
    price_df = pd.DataFrame(close_series_map).dropna(how="any")
    print(f"Time-series aligned successfully. Dimension: {price_df.shape[0]} trading days x {price_df.shape[1]} symbols.")

    # 3. Mathematical Transformation: Daily Log Returns
    # Utilizing log returns to stabilize statistical skewness across changing macro cycles
    log_returns = np.log(price_df / price_df.shift(1)).dropna()

    # 4. Generate Pearson Product-Moment Correlation Matrix
    correlation_matrix = log_returns.corr(method="pearson")

    # 5. Render Visual Output Layer (Dark-Themed Layout Specs)
    fig = px.imshow(
        correlation_matrix,
        text_auto=".2f",
        color_continuous_scale="RdBu_r", # Diverging color scheme emphasizing polarity
        zmin=-1.0,
        zmax=1.0,
        labels=dict(x="Asset Identifier", y="Asset Identifier", color="Pearson r"),
        title="QuantIQ | Core NIFTY50 Matrix — 1-Year Daily Return Correlation Profile"
    )

    fig.update_layout(
        height=700,
        width=750,
        coloraxis_colorbar=dict(
            title="Correlation Profile",
            thicknessmode="pixels", thickness=15,
            lenmode=" Imperial", len=0.8
        ),
        margin=dict(l=50, r=50, b=50, t=80)
    )

    fig.show()

# ── RUNTIME EXECUTION GATEWAY ────────────────────────────────────────────────
# UNCOMMENT AND RUN AS SOON AS ALL THE 12 INDIVIDUAL STOCK MODULES ARE MERGED
# generate_correlation_matrix(TICKERS, period=PERIOD, interval=INTERVAL)

### Observations

_TODO (RS + GT): Write 3–5 paragraphs on correlation patterns — which sectors move together, which are uncorrelated, and what this implies for portfolio diversification._